# Style Prefix Demo (GPT-2 Nano)
This notebook installs Levanter from the `feat/style-prefix-token` branch,
creates a tiny chat dataset with `style` labels, inspects the resulting
prefix tokens and loss mask, and runs a short GPT-2 "nano" training loop.

In [ ]:
%%bash

set -euo pipefail

echo 'Installing base dependencies (datasets, wandb, draccus, jax, jaxlib) and pinning protobuf...\n'
pip install --quiet --no-deps --force-reinstall "protobuf<5"
pip install --quiet \
  datasets wandb draccus jax jaxlib
echo "Base dependencies installed.\n"

REPO_DIR=/content/levanter

if [ -d "$REPO_DIR/.git" ]; then
  echo "Found existing repo at $REPO_DIR. Syncing feat/style-prefix-token...
"
  git -C "$REPO_DIR" fetch origin feat/style-prefix-token
  git -C "$REPO_DIR" checkout feat/style-prefix-token
  echo 'Resetting local branch to origin/feat/style-prefix-token...'
  git -C "$REPO_DIR" reset --hard origin/feat/style-prefix-token
  echo 'Cleaning untracked files...'
  git -C "$REPO_DIR" clean -fd
else
  echo "No repo found. Cloning fresh copy to $REPO_DIR...
"
  rm -rf "$REPO_DIR"
  git clone https://github.com/chris544460/levanter.git "$REPO_DIR"
  git -C "$REPO_DIR" checkout feat/style-prefix-token
  git -C "$REPO_DIR" reset --hard origin/feat/style-prefix-token
fi

echo 'Ensuring style demo dataset exists...'
mkdir -p "$REPO_DIR/data/style_demo"
cat <<'EOF' > "$REPO_DIR/data/style_demo/train.jsonl"
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "Who wrote the Odyssey?"}, {"role": "assistant", "content": "Homer wrote the Odyssey."}], "style": "wiki"}
{"messages": [{"role": "system", "content": "style=books"}, {"role": "user", "content": "Recommend a fantasy series."}, {"role": "assistant", "content": "Try The Wheel of Time by Robert Jordan."}], "style": "books"}
{"messages": [{"role": "system", "content": "style=news"}, {"role": "user", "content": "Summarize today's headlines."}, {"role": "assistant", "content": "Major markets rallied; new policies were announced."}], "style": "news"}
{"messages": [{"role": "system", "content": "style=wiki"}, {"role": "user", "content": "What is photosynthesis?"}, {"role": "assistant", "content": "Photosynthesis converts light energy into chemical energy."}], "style": "wiki"}
EOF
echo 'Style demo dataset written to $REPO_DIR/data/style_demo/train.jsonl.'



In [ ]:
%%bash
cd /content/levanter
pip uninstall -y levanter || true
pip install --no-deps -e .
pip install --quiet --no-deps --force-reinstall 'protobuf<5'
pip show protobuf


In [ ]:
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer

data_path = Path('/content/levanter/data/style_demo/train.jsonl')
if not data_path.exists():
    raise FileNotFoundError(f'Style demo dataset not found at {data_path}. Ensure the setup cell has run.')

dataset = load_dataset('json', data_files={'train': str(data_path)}, split='train')
entries = dataset.to_list()

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'additional_special_tokens': ['<style>', '</style>']})

print(f'Loaded {len(entries)} chat entries.')
print(f'Tokenizer vocab size (with added specials): {tokenizer.vocab_size + len(tokenizer.added_tokens_decoder)}')


In [ ]:
import os
import sys
from pathlib import Path

_env_srcs = []
if 'LEVANTER_SRC_DIR' in os.environ:
    _env_srcs.append(Path(os.environ['LEVANTER_SRC_DIR']))
if 'LEVANTER_REPO' in os.environ:
    _env_srcs.append(Path(os.environ['LEVANTER_REPO']) / 'src')

_default_roots = [
    Path.cwd(),
    *list(Path.cwd().parents[:2]),
    Path('/content/levanter'),
    Path('/workspace/levanter'),
    Path('/kaggle/working/levanter'),
]

_candidates = []
for root in _env_srcs + _default_roots:
    if root is None:
        continue
    root = root.resolve()
    if root.name == 'src':
        _candidates.append(root)
    else:
        _candidates.append(root / 'src')

for _src in _candidates:
    if _src.exists():
        if str(_src) not in sys.path:
            sys.path.append(str(_src))
        break
else:
    raise ModuleNotFoundError('Could not locate Levanter src directory. Set LEVANTER_SRC_DIR or ensure repo is cloned.')

from levanter.data.text import ChatLmDatasetFormat, StylePrefixConfig, preprocessor_for_format

if 'tokenizer' not in globals() or 'entries' not in globals():
    raise RuntimeError('Run the tokenizer/data setup cell before formatting the dataset.')

chat_template = (
    "{%- for message in messages -%}\n"
    "{%- if message['role'] == 'assistant' -%}\n"
    "{% generation %}assistant: {{ message['content'] }}\n"
    "{% endgeneration %}\n"
    "{%- elif message['role'] == 'user' -%}\n"
    "user: {{ message['content'] }}\n"
    "{%- elif message['role'] == 'system' -%}\n"
    "system: {{ message['content'] }}\n"
    "{%- else -%}\n"
    "{{ message['role'] }}: {{ message['content'] }}\n"
    "{%- endif -%}\n"
    "{%- endfor -%}\n"
    "{%- if add_generation_prompt %}\n"
    "{% generation %}assistant: \n"
    "{% endgeneration %}\n"
    "{%- endif %}\n"
)

format_cfg = ChatLmDatasetFormat(
    messages_field="messages",
    single_turn=False,
    chat_template=chat_template,
    pack=True,
    mask_user_turns=False,
    style_prefix=StylePrefixConfig(
        prefix_token='<style>',
        suffix_token='</style>',
        style_field="style",
    ),
)

processor = preprocessor_for_format(format_cfg, tokenizer)
processed = processor(entries[:2])

for idx, example in enumerate(processed):
    print(f"Example {idx}")
    print("input_ids:", example["input_ids"][:20])
    print("assistant_masks:", example["assistant_masks"][:20])
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"][:20])
    print("tokens:", tokens)
    print()


In [ ]:
%%bash
cd /content/levanter
PYTHONPATH=$PWD python -m levanter.main.train_lm --config_path config/gpt2_nano_style.yaml